# Civix SDK Pipeline Quickstart

This notebook fetches Vancouver business licences through the public SDK, previews normalized records, exports JSON artifacts, and builds a small Altair chart. It hits live public APIs and writes generated files under `notebooks/out/` when run from the repo root, or `out/` when run from inside the `notebooks/` directory.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import altair as alt
import pandas as pd
import vl_convert as vlc
from IPython.display import HTML

from civix.core.quality.models.fields import MappedField
from civix.domains.business_licences.models import BusinessLicence
from civix.infra.exporters.json import write_snapshot
from civix.sdk import Civix


def default_output_dir() -> Path:
    if Path.cwd().name == "notebooks":
        return Path("out/sdk_pipeline_quickstart")

    return Path("notebooks/out/sdk_pipeline_quickstart")


OUTPUT_DIR = default_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Fetch And Preview

In [2]:
async with Civix() as client:
    product = client.datasets.ca.business_licences.licence.vancouver
    result = await client.fetch(product)
    snapshot = result.snapshot

    preview = []
    quality_counts = Counter()
    unmapped_counts = Counter()

    async for paired in result.records:
        record = paired.mapped.record

        if len(preview) < 5:
            preview.append(record)

        for field_name in record.__class__.model_fields:
            value = getattr(record, field_name)
            if isinstance(value, MappedField):
                quality_counts[value.quality.value] += 1

        unmapped_counts.update(paired.mapped.report.unmapped_source_fields)

snapshot.model_dump(mode="json")

{'snapshot_id': 'vancouver-open-data:business-licences:2026-05-06T06:15:59.706166+00:00',
 'source_id': 'vancouver-open-data',
 'dataset_id': 'business-licences',
 'jurisdiction': {'country': 'CA', 'region': 'BC', 'locality': 'Vancouver'},
 'fetched_at': '2026-05-06T06:15:59.706166Z',
 'record_count': 201323,
 'source_url': 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/business-licences/exports/jsonl',
 'fetch_params': None,
 'content_hash': None}

In [3]:
pd.DataFrame(
    {
        "business_name": record.business_name.value,
        "licence_number": record.licence_number.value,
        "status": record.status.value.value if record.status.value else None,
        "neighbourhood": record.neighbourhood.value,
    }
    for record in preview
)

,business_name,licence_number,status,neighbourhood
0,Elmwood Cannabis Corporation,24-145433,active,Mount Pleasant
1,Biktrix Enterprises Inc,24-145435,active,Renfrew-Collingwood
2,One Wednesday Home Goods Inc,24-145437,active,Riley Park
3,Pisces Signage Ltd,24-145438,active,Hastings-Sunrise
4,Barngey’s Inc,24-145440,active,Hastings-Sunrise


In [4]:
pd.DataFrame(quality_counts.items(), columns=["quality", "count"]).sort_values(
    "count", ascending=False
)

,quality,count
1,standardized,646627
0,direct,588164
2,derived,359651
3,not_provided,217465


## Export And Chart

In [5]:
async with Civix() as client:
    product = client.datasets.ca.business_licences.licence.vancouver
    result = await client.fetch(product)
    manifest = await write_snapshot(
        result,
        output_dir=OUTPUT_DIR,
        record_type=BusinessLicence,
    )

snapshot_dir = OUTPUT_DIR / manifest.snapshot_id
records = [
    BusinessLicence.model_validate_json(line)
    for line in (snapshot_dir / "records.jsonl").read_text().splitlines()
    if line
]

manifest.model_dump(mode="json")

{'snapshot_id': 'vancouver-open-data:business-licences:2026-05-06T06:16:42.421111+00:00',
 'source_id': 'vancouver-open-data',
 'dataset_id': 'business-licences',
 'jurisdiction': {'country': 'CA', 'region': 'BC', 'locality': 'Vancouver'},
 'fetched_at': '2026-05-06T06:16:42.421111Z',
 'record_count': 201323,
 'mapper': {'mapper_id': 'vancouver-business-licences', 'version': '0.1.0'},
 'files': [{'filename': 'schema.json',
   'sha256': '7dc653198f150df5c3c140524d2f2c9e2d32529a96a811744fde128a8670c102',
   'byte_count': 13906},
  {'filename': 'records.jsonl',
   'sha256': 'dec658d4fbd95826f6c1fb745012fc7e96b98059e2b83af331cf98644ba37a03',
   'byte_count': 301213522},
  {'filename': 'reports.jsonl',
   'sha256': '71751938b371adbab0be65199688c09789783a5f2b228ce519ace3232b1f69ef',
   'byte_count': 35835494}],
 'mapping_summary': {'quality_counts': {'direct': 588164,
   'standardized': 646627,
   'derived': 359651,
   'not_provided': 217465},
  'unmapped_source_fields_total': 1207938,
  'co

In [6]:
rows = []
for record in records:
    rows.append(
        {
            "neighbourhood": record.neighbourhood.value or "Not provided",
            "status": record.status.value.value if record.status.value else "not_provided",
            "licences": 1,
        }
    )

chart_data = pd.DataFrame(rows)
top_neighbourhoods = (
    chart_data.groupby("neighbourhood")["licences"]
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .index
)
chart_data = (
    chart_data[chart_data["neighbourhood"].isin(top_neighbourhoods)]
    .groupby(["neighbourhood", "status"], as_index=False)["licences"]
    .sum()
)

chart = (
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        x=alt.X("sum(licences):Q", title="Licences"),
        y=alt.Y("neighbourhood:N", sort="-x", title="Neighbourhood"),
        color=alt.Color("status:N", title="Status"),
        tooltip=["neighbourhood:N", "status:N", "sum(licences):Q"],
    )
    .properties(
        title="Vancouver business licences by neighbourhood and status",
        width=820,
        height=420,
    )
)

svg = vlc.vegalite_to_svg(chart.to_dict())
HTML(svg)